# Spike 03 — English OCR with Chandra 2 (Datalab Cloud API)

**Goal:** Extract structured English text from the Tranquil City English page images using Chandra 2 via the Datalab REST API.

**Time box:** 1 hour

**Question to answer:** Does Chandra produce well-structured Markdown (especially tables) from English report pages?

**Done when:** `.md` files in `ocr_output/english/` contain readable text with tables preserved as Markdown.

---

### Why Chandra 2 for English?

| Model | Overall Average (90 langs) | Arabic |
|---|---|---|
| **Chandra 2** | **72.7%** ✅ | 68.4% |
| Gemini 2.5 Flash | 60.8% | 84.4% |

Chandra wins on overall quality and outputs structured Markdown that preserves tables, column headers, and multi-column layouts — critical for urban statistics reports.

---

### How the API works

```
Your PNG image
      ↓  (multipart/form-data upload)
https://www.datalab.to/api/v1/ocr
      ↓  (JSON response)
{ "markdown": "# Extracted text..." }
```

No GPU. No model loading. No RAM issues. Just HTTP.

### API Key Setup
1. Go to [datalab.to](https://www.datalab.to)
2. Sign up → get **\$5 free credits**
3. Copy API key → add to `.env`: `CHANDRA_API_KEY=your_key_here`

## Step 1 — Imports & Setup

In [ ]:
import requests
from dotenv import load_dotenv
from pathlib import Path
import os
import time

load_dotenv(dotenv_path="../.env")

API_KEY  = os.getenv("CHANDRA_API_KEY")
API_URL  = "https://www.datalab.to/api/v1/ocr"
HEADERS  = {"Authorization": f"Bearer {API_KEY}"}

if not API_KEY:
    raise ValueError("CHANDRA_API_KEY not found in .env file — get it from datalab.to")

print("✅ Datalab API client ready")
print(f"   Endpoint: {API_URL}")

## Step 2 — OCR Function

In [ ]:
def ocr_page(image_path: str, output_format: str = "markdown") -> str:
    """
    Send a PNG page image to Datalab's Chandra OCR API.
    Returns extracted text as a Markdown string.

    Args:
        image_path   : path to the PNG file
        output_format: 'markdown' (default) or 'html'
                       Use 'html' if tables don't render well in markdown
    """
    with open(image_path, "rb") as f:
        files = {"file": (Path(image_path).name, f, "image/png")}
        data  = {"output_format": output_format}

        response = requests.post(
            API_URL,
            headers = HEADERS,
            files   = files,
            data    = data,
            timeout = 60
        )

    if response.status_code != 200:
        raise RuntimeError(f"API error {response.status_code}: {response.text[:300]}")

    result = response.json()

    # Extract text — try common response keys
    text = (
        result.get("markdown") or
        result.get("text")     or
        result.get("content")  or
        str(result)
    )

    return text


print("✅ ocr_page() defined")

## Step 3 — Test on a Single Page First

Always validate one page before running all 14.  
This also lets us see the raw API response structure so we know which key holds the text.

In [ ]:
en_images = sorted(Path("../data/images_en").glob("*.png"))

if not en_images:
    print("❌ No English images found — run Spike 01 first")
else:
    print(f"Found {len(en_images)} English page images")

    # Test on page 6 — more likely to have real content than the cover
    test_page = en_images[5] if len(en_images) > 5 else en_images[0]
    print(f"Testing on   : {test_page.name}")
    print()

    # Show the raw response first so we understand the structure
    with open(str(test_page), "rb") as f:
        raw_response = requests.post(
            API_URL,
            headers = HEADERS,
            files   = {"file": (test_page.name, f, "image/png")},
            data    = {"output_format": "markdown"},
            timeout = 60
        )

    print(f"Status code  : {raw_response.status_code}")
    print(f"Response keys: {list(raw_response.json().keys())}")
    print()

    # Now run through our function
    test_result = ocr_page(str(test_page))
    print("=== OCR OUTPUT (first 1000 chars) ===")
    print(test_result[:1000])
    print()
    print(f"Total characters: {len(test_result)}")

## Step 4 — Check Table Preservation

Urban reports are full of statistics tables. Verify one page that likely contains a table.

In [ ]:
# Try pages 8-12 — deeper in the report usually has data tables
# Change this index if you know which page has a table
TABLE_PAGE_IDX = 9   # 0-indexed → page 10

if len(en_images) > TABLE_PAGE_IDX:
    table_page = en_images[TABLE_PAGE_IDX]
    print(f"Testing table extraction on: {table_page.name}")

    md_result   = ocr_page(str(table_page), output_format="markdown")
    html_result = ocr_page(str(table_page), output_format="html")

    print("\n=== MARKDOWN FORMAT ===")
    print(md_result[:800])

    print("\n=== HTML FORMAT ===")
    print(html_result[:800])

    print("\n=== WHICH IS BETTER? ===")
    print(f"Markdown tables (|) : {'✅ Yes' if '|' in md_result else '❌ No'}")
    print(f"HTML tables (<table>): {'✅ Yes' if '<table' in html_result.lower() else '❌ No'}")
    print()
    print("→ Set BEST_FORMAT in Step 5 to whichever preserves tables better")

## Step 5 — Run OCR on All 14 Pages

In [ ]:
OUTPUT_DIR  = Path("../ocr_output/english")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_FORMAT = "markdown"   # ← change to "html" if Step 4 showed better table output

en_images   = sorted(Path("../data/images_en").glob("*.png"))[:14]
results     = {}

print(f"Processing {len(en_images)} pages via Datalab API...\n")

for i, img_path in enumerate(en_images):
    out_path = OUTPUT_DIR / img_path.with_suffix(".md").name

    if out_path.exists():
        print(f"⏭  Skipping {img_path.name} — already processed")
        results[img_path.stem] = out_path.read_text(encoding="utf-8")
        continue

    print(f"[{i+1}/{len(en_images)}] {img_path.name}...", end=" ", flush=True)

    try:
        text = ocr_page(str(img_path), output_format=BEST_FORMAT)
        out_path.write_text(text, encoding="utf-8")
        results[img_path.stem] = text
        print(f"✅ {len(text)} chars")
    except Exception as e:
        print(f"❌ {e}")

    # Small delay between calls
    if i < len(en_images) - 1:
        time.sleep(1)

print(f"\nDone — {len(results)} pages processed")

## Step 6 — Side-by-Side Quality Check

In [ ]:
from IPython.display import display, Image as IPImage

OUTPUT_DIR = Path("../ocr_output/english")
en_images  = sorted(Path("../data/images_en").glob("*.png"))
md_files   = sorted(OUTPUT_DIR.glob("*.md"))

PAGE_TO_CHECK = 6   # change to inspect different pages

if len(md_files) >= PAGE_TO_CHECK and len(en_images) >= PAGE_TO_CHECK:
    img_file  = en_images[PAGE_TO_CHECK - 1]
    text_file = md_files[PAGE_TO_CHECK - 1]

    print(f"=== ORIGINAL IMAGE: {img_file.name} ===")
    display(IPImage(filename=str(img_file), width=650))

    print(f"\n=== CHANDRA OCR OUTPUT: {text_file.name} ===")
    content = text_file.read_text(encoding="utf-8")
    print(content[:2000])
    print(f"\n[{len(content)} total chars]")
else:
    print(f"Only {len(md_files)} pages processed — adjust PAGE_TO_CHECK")

## Step 7 — Spike Summary

In [ ]:
OUTPUT_DIR = Path("../ocr_output/english")
en_images  = sorted(Path("../data/images_en").glob("*.png"))[:14]
md_files   = sorted(OUTPUT_DIR.glob("*.md"))

total_chars  = sum(f.read_text(encoding="utf-8").__len__() for f in md_files)
tables_found = sum(1 for f in md_files if "|" in f.read_text(encoding="utf-8"))

print("=" * 50)
print("SPIKE 03 — RESULTS")
print("=" * 50)
print(f"OCR engine           : Chandra 2 (Datalab Cloud API)")
print(f"Pages processed      : {len(md_files)} / {len(en_images)}")
print(f"Total chars extracted: {total_chars:,}")
print(f"Avg chars per page   : {total_chars // max(len(md_files), 1):,}")
print(f"Pages with tables    : {tables_found}")
print()

if len(md_files) == len(en_images) and total_chars > 1000:
    print("✅ SPIKE PASSED — English OCR output ready for Spike 04 (PageIndex)")
    print(f"   Files in: {OUTPUT_DIR.resolve()}")
else:
    print(f"⚠️  SPIKE INCOMPLETE — {len(en_images) - len(md_files)} pages missing")